# HV-012 — Monte Carlo Uncertainty

**Technical gate:** MC-001  
**Analytical baseline:** HV-011 / MU-001

This notebook validates the analytical uncertainty propagation and then tests sensitivity to the observed finite Direction × Run state structure.

## 1. Engineering Question

> Does Monte Carlo propagation reproduce the HV-011 analytical uncertainty budget, and how does distribution shape change when the point-state component is represented by the observed Direction × Run support instead of a Gaussian approximation?

Two frozen models are used:

- **Model A:** analytical-equivalent Gaussian state component;
- **Model B:** variance-matched empirical state bootstrap.

## 2. Reproducible Setup

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

current_dir = Path.cwd()
candidate_roots = [current_dir, *current_dir.parents]

project_root = next(
    (
        path
        for path in candidate_roots
        if (path / "reports" / "tables" / "hv011_uncertainty_summary.csv").exists()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "Could not locate project root containing HV-011 uncertainty evidence."
    )

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.monte_carlo_analysis import (
    MonteCarloConfig,
    build_empirical_state_support,
    build_model_comparison,
    load_hv011_inputs,
    run_monte_carlo,
    validate_mc_gate,
)

tables_dir = project_root / "reports" / "tables"
figures_dir = project_root / "reports" / "figures"
tables_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

mc_config = MonteCarloConfig(seed=7140, trials=250_000)

print(f"Project root: {project_root}")
print(f"Monte Carlo seed: {mc_config.seed}")
print(f"Trials per point per model: {mc_config.trials:,}")

## 3. MC-01 — HV-011 Input Audit

HV-012 consumes only curated HV-011 analyst-visible evidence.

Expected:

- four evaluation points;
- four uncertainty components per point;
- four Test_ID state observations per point.

In [ ]:
hv011_summary_df, hv011_budget_df, hv011_evidence_df = load_hv011_inputs(
    project_root
)

assert hv011_summary_df["Point_ID"].nunique() == 4
assert len(hv011_summary_df) == 4
assert len(hv011_budget_df) == 16
assert (
    hv011_evidence_df.groupby("Point_ID").size().eq(4).all()
)

print("HV-011 point summary")
display(
    hv011_summary_df[
        [
            "Point_ID",
            "RH_Percent",
            "Temperature_C",
            "Point_Residual_Bias",
            "U_State",
            "Combined_Standard_Uncertainty",
            "Expanded_Uncertainty",
        ]
    ].round(6)
)

print("MC-01 HV-011 input audit: PASS")

## 4. MC-02 — Empirical State Support

For Model B, the four observed Test_ID residual means at each point are:

1. centered by subtracting the point mean;
2. rescaled to match the frozen HV-011 `u_state`;
3. sampled with equal probability.

This preserves the observed UP/DOWN and Run state pattern while keeping the same variance budget.

In [ ]:
mc_state_support_df = build_empirical_state_support(
    hv011_summary_df,
    hv011_evidence_df,
)

support_audit = (
    mc_state_support_df
    .groupby("Point_ID")
    .agg(
        N_States=("MC_State_Support", "size"),
        Support_Mean=("MC_State_Support", "mean"),
        Support_Pop_SD=(
            "MC_State_Support",
            lambda x: np.std(x.to_numpy(dtype=float), ddof=0),
        ),
        Target_U_State=("Target_U_State", "first"),
    )
    .reset_index()
)

assert (support_audit["N_States"] == 4).all()
assert np.allclose(support_audit["Support_Mean"], 0.0, atol=1e-12)
assert np.allclose(
    support_audit["Support_Pop_SD"],
    support_audit["Target_U_State"],
    rtol=1e-12,
    atol=1e-12,
)

display(mc_state_support_df.round(6))
print("State-support audit")
display(support_audit.round(6))

print("MC-02 empirical state support: PASS")

## 5. MC-03 — Run Frozen Monte Carlo Models

Common components:

```text
Repeatability ~ Normal(0, u_repeat_mean)
Reference     ~ Normal(0, u_ref_cal)
Stability     ~ Uniform(-a_stability, +a_stability)
```

State component:

```text
Model A: Normal(0, u_state)
Model B: variance-matched empirical bootstrap
```

In [ ]:
mc_summary_df, mc_samples = run_monte_carlo(
    summary_df=hv011_summary_df,
    state_support_df=mc_state_support_df,
    config=mc_config,
)

assert len(mc_summary_df) == 8
assert set(mc_summary_df["Model"]) == {
    "A_ANALYTICAL_EQUIVALENT",
    "B_EMPIRICAL_STATE",
}

display(
    mc_summary_df[
        [
            "Point_ID",
            "Model",
            "MC_Mean",
            "MC_SD",
            "Relative_SD_Error_vs_Analytical",
            "Q025",
            "Median",
            "Q975",
            "Coverage_Inside_PlusMinus_2uc",
            "Skewness",
            "Excess_Kurtosis",
        ]
    ].round(6)
)

print("MC-03 frozen Monte Carlo models executed: PASS")

## 6. MC-04 — Numerical Validation Checks

Frozen tolerances:

### Model A

```text
|MC_SD - analytical u_c| / u_c <= 1%
|MC_Mean| <= 0.01 %RH
```

### Model B

```text
|MC_SD - analytical u_c| / u_c <= 1.5%
|MC_Mean| <= 0.01 %RH
```

Model B is not required to reproduce Gaussian interval shape.

In [ ]:
mc_gate_checks_df = validate_mc_gate(mc_summary_df)

display(mc_gate_checks_df)

assert mc_gate_checks_df["Gate_Check_PASS"].all()

model_a_checks = mc_gate_checks_df.loc[
    mc_gate_checks_df["Model"].eq("A_ANALYTICAL_EQUIVALENT")
]
model_b_checks = mc_gate_checks_df.loc[
    mc_gate_checks_df["Model"].eq("B_EMPIRICAL_STATE")
]

assert len(model_a_checks) == 4
assert len(model_b_checks) == 4

print("MC-04 numerical variance/centering validation: PASS")

## 7. MC-05 — Analytical Interval vs Monte Carlo Distribution

The analytical HV-011 interval is:

```text
± 2 u_c
```

This section compares it with the actual central 95% Monte Carlo interval under both distribution models.

In [ ]:
mc_model_comparison_df = build_model_comparison(mc_summary_df)

display(mc_model_comparison_df.round(6))

print("MC-05 analytical/Monte Carlo interval comparison: PASS")

## 8. MC-06 — Residual Bias Kept Separate

Consistent with HV-011, Monte Carlo uncertainty error is centered at zero.

For descriptive total-error envelopes only:

```text
Total_Error = Point_Residual_Bias + MC_Uncertainty_Error
```

No residual-bias correction is introduced here.

In [ ]:
total_error_rows = []

for _, row in mc_summary_df.iterrows():
    point_id = row["Point_ID"]
    model = row["Model"]
    bias = float(row["Point_Residual_Bias"])
    values = mc_samples[(point_id, model)] + bias

    q025, q50, q975 = np.quantile(values, [0.025, 0.50, 0.975])

    total_error_rows.append(
        {
            "Point_ID": point_id,
            "Model": model,
            "Residual_Bias": bias,
            "Total_Error_Q025": q025,
            "Total_Error_Median": q50,
            "Total_Error_Q975": q975,
            "Probability_Within_Public_Static_Accuracy_Limit": np.mean(
                np.abs(values) <= 2.0
            ),
        }
    )

mc_total_error_df = pd.DataFrame(total_error_rows)

display(mc_total_error_df.round(6))

print("MC-06 residual-bias separation: PASS")

## 9. MC-07 — Distribution-Shape Interpretation Diagnostics

Model A should be close to symmetric with a central 95% half-width near, but not necessarily equal to, `2u_c`.

Model B preserves four finite Direction × Run states. Its variance is matched to HV-011, but its tails may be materially different.

This is the key distinction:

```text
same variance does not imply same coverage interval
```

In [ ]:
shape_diagnostics_df = mc_summary_df[
    [
        "Point_ID",
        "Model",
        "Analytical_2uc",
        "Central95_Symmetric_HalfWidth",
        "Coverage_Inside_PlusMinus_2uc",
        "Skewness",
        "Excess_Kurtosis",
    ]
].copy()

shape_diagnostics_df["MC95_to_Analytical2uc_Ratio"] = (
    shape_diagnostics_df["Central95_Symmetric_HalfWidth"]
    / shape_diagnostics_df["Analytical_2uc"]
)

display(shape_diagnostics_df.round(6))

print("MC-07 distribution-shape diagnostics: PASS")

## 10. Curated Figures

In [ ]:
# Central-95 half-width compared with analytical 2*u_c
plot_df = mc_summary_df.copy()

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(hv011_summary_df["Point_ID"].nunique())
width = 0.28

for j, model in enumerate(
    ["A_ANALYTICAL_EQUIVALENT", "B_EMPIRICAL_STATE"]
):
    g = (
        plot_df.loc[plot_df["Model"].eq(model)]
        .sort_values("Point_ID")
    )
    ax.bar(
        x + (j - 0.5) * width,
        g["Central95_Symmetric_HalfWidth"],
        width=width,
        label=model,
    )

analytical = (
    plot_df.loc[
        plot_df["Model"].eq("A_ANALYTICAL_EQUIVALENT")
    ]
    .sort_values("Point_ID")
)

ax.plot(
    x,
    analytical["Analytical_2uc"],
    marker="o",
    label="HV-011 analytical 2*u_c",
)

ax.set_xticks(x)
ax.set_xticklabels(analytical["Point_ID"])
ax.set_ylabel("95% half-width / analytical 2u_c (%RH)")
ax.set_xlabel("Evaluation point")
ax.set_title("HV-012 Analytical vs Monte Carlo Interval Scale")
ax.legend()
fig.tight_layout()
fig.savefig(
    figures_dir / "hv012_mc_interval_comparison.png",
    dpi=160,
    bbox_inches="tight",
)
plt.show()

# Representative point distribution: UNC-002
point_id = "UNC-002"
a_values = mc_samples[(point_id, "A_ANALYTICAL_EQUIVALENT")]
b_values = mc_samples[(point_id, "B_EMPIRICAL_STATE")]

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(
    a_values,
    bins=100,
    density=True,
    histtype="step",
    linewidth=1.6,
    label="Model A — analytical-equivalent",
)
ax.hist(
    b_values,
    bins=100,
    density=True,
    histtype="step",
    linewidth=1.6,
    label="Model B — empirical state",
)

u_c_rep = float(
    hv011_summary_df.loc[
        hv011_summary_df["Point_ID"].eq(point_id),
        "Combined_Standard_Uncertainty",
    ].iloc[0]
)
ax.axvline(-2.0 * u_c_rep, linestyle="--", label="-2u_c")
ax.axvline(+2.0 * u_c_rep, linestyle="--", label="+2u_c")

ax.set_xlabel("Zero-centered uncertainty error (%RH)")
ax.set_ylabel("Density")
ax.set_title("HV-012 Representative Monte Carlo Distribution — UNC-002")
ax.legend()
fig.tight_layout()
fig.savefig(
    figures_dir / "hv012_mc_representative_distribution.png",
    dpi=160,
    bbox_inches="tight",
)
plt.show()

print("Curated HV-012 figures exported.")

## 11. Curated Evidence Export

In [ ]:
mc_summary_df.to_csv(
    tables_dir / "hv012_mc_summary.csv",
    index=False,
)
mc_state_support_df.to_csv(
    tables_dir / "hv012_mc_state_support.csv",
    index=False,
)
mc_model_comparison_df.to_csv(
    tables_dir / "hv012_mc_model_comparison.csv",
    index=False,
)

print("Curated HV-012 tables exported.")

## 12. Initial MC-001 Checkpoint

A clean-kernel Run All should establish:

- MC-01 HV-011 input audit PASS;
- MC-02 empirical-state support PASS;
- MC-03 both frozen models executed;
- MC-04 numerical variance/centering validation PASS;
- MC-05 analytical/MC interval comparison PASS;
- MC-06 residual-bias separation PASS;
- MC-07 distribution-shape diagnostics PASS;
- three curated CSV tables;
- two curated figures.

The next step is engineering interpretation and formal MC-001 gate closure.